In [1]:
# Importing libraries and setup
from dotenv import load_dotenv
import requests
from agents import Runner, trace, Agent, function_tool, ModelSettings
from agents.extensions.visualization import draw_graph
from openai.types.responses import ResponseTextDeltaEvent
import os
import asyncio
import smtplib
from email.message import EmailMessage
load_dotenv(override=True)

MODEL_NAME = 'gpt-4o-mini'

In [2]:
# Configurations
EMAIL_ADDRESS = os.getenv("EMAIL_ADDRESS")
EMAIL_SMTP_SERVER = os.getenv("EMAIL_SMTP_SERVER")
EMAIL_APP_PASSWORD = os.getenv("EMAIL_APP_PASSWORD")

In [3]:
# Function to send the email
def send_email(subject, text_body, html_body):
    msg = EmailMessage()
    msg['From'] = EMAIL_ADDRESS
    msg['To'] = EMAIL_ADDRESS
    msg['Subject'] = subject 
    msg.set_content(text_body)
    msg.add_alternative(html_body, subtype='html')

    with smtplib.SMTP(EMAIL_SMTP_SERVER, 587) as server:
        server.starttls()
        server.login(EMAIL_ADDRESS, EMAIL_APP_PASSWORD)
        server.send_message(msg)

In [4]:
# Testing the function
send_email("Testing testing 123", "Fingers crossed..", "<html><body><strong>Fingers</strong> crossed..</body></html>")

### Push notification setup

In [5]:
# Configuration
pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

# function to send push notification
def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

In [6]:
USE_EMAIL = True # if you want to send notification instead of email

def send_message(subject, text_body, html_body):
    if USE_EMAIL:
        send_email(subject, text_body, html_body)
    else:
        push(f"Subject: {subject}\n\n{text_body}")

In [7]:
# Testing
send_message("Big news", "Communications are a go!", "<html><body>Communications are a <strong>go!</strong></body></html>")

## Agent Orchestration

There are 2 models for Agent Orchestration  
by code and by LLMs.  
**By code**: more predictable and deterministic.  
**By LLMs**: more powerful.  

An excellent write-up is here:  
https://openai.github.io/openai-agents-python/multi_agent/

### Orchestration by code

In [8]:
intro = """
You are a sales agent working for ComplAI,
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI.
You write emails.
"""

instructions1 = intro + "Your email style is professional, serious, with gravitas and credibility"
instructions2 = intro + "Your email style is witty, engagging, and humorous"
instructions3 = intro + "Your email style is concise, to the point, in the style of a busy senior executive."

In [9]:
print(instructions1)


You are a sales agent working for ComplAI,
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI.
You write emails.
Your email style is professional, serious, with gravitas and credibility


In [10]:
# Creating different sales agents
sales_agent1 = Agent(name='Professional Sales Agent', instructions=instructions1, model=MODEL_NAME)
sales_agent2 = Agent(name="Humorous Sales Agent", instructions=instructions2, model=MODEL_NAME)
sales_agent3 = Agent(name="Executive Sales Agent", instructions=instructions3, model=MODEL_NAME)


In [11]:
from IPython.display import display, Markdown
from IPython.display import clear_output
from openai.types.responses import ResponseTextDeltaEvent

result = Runner.run_streamed(sales_agent1, input="Write a cold sales email")

response = ""
async for event in result.stream_events():

    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        response += event.data.delta
        
        clear_output(wait=True)
        display(Markdown(response))

Subject: Elevate Your SOC 2 Compliance Strategy with AI-Powered Solutions

Dear [Recipient's Name],

I hope this message finds you well. My name is [Your Name], and I represent ComplAI, a leader in providing innovative SaaS solutions tailored for SOC 2 compliance.

In today's rapidly evolving regulatory landscape, ensuring SOC 2 compliance is not just a necessity—it's a cornerstone of trust with your clients. However, the complexities involved can be daunting and resource-intensive. That’s where our AI-driven tool comes into play.

At ComplAI, we specialize in simplifying the compliance process through advanced technology that automates documentation, identifies gaps, and prepares your organization for audits seamlessly. Our solution not only enhances efficiency but also significantly reduces the risk of non-compliance penalties.

I would welcome the opportunity to discuss how ComplAI can help your organization streamline its compliance efforts and fortify your audit readiness. Would you be available for a brief call in the coming days?

Thank you for considering this opportunity. I look forward to the possibility of working together to strengthen your compliance strategy.

Warm regards,

[Your Name]  
[Your Position]  
ComplAI  
[Your Phone Number]  
[Your Email Address]  
[Company Website]  

In [12]:
# Running all three agents and storing their results

message = "Write a cold sales email"

with trace("Parallel cold emails"):
    results = await asyncio.gather(
        Runner.run(sales_agent1, message),
        Runner.run(sales_agent2, message),
        Runner.run(sales_agent3, message)
    )

outputs = [result.final_output for result in results]

for output in outputs:
    display(Markdown(output))
    print(f"\n{'--'* 50}\n")

Subject: Elevate Your SOC 2 Compliance with AI-Powered Solutions

Dear [Recipient's Name],

I hope this message finds you well.

In an increasingly regulated landscape, achieving and maintaining SOC 2 compliance has become paramount for businesses like yours. At ComplAI, we recognize the challenges organizations face in managing compliance requirements and preparing for audits. 

Our AI-powered SaaS tool is designed to streamline the compliance process, helping you not only to achieve SOC 2 certification but to sustain it with minimal operational disruption. We offer a robust solution that automates documentation, tracks controls, and provides real-time insights to reduce the burden of audit preparation.

By partnering with ComplAI, you can:

- Reduce compliance costs and time by up to 70%.
- Enhance data security posture with ongoing monitoring and updates.
- Ensure preparedness for audits with thorough, automated reporting.

I would welcome the opportunity to discuss how ComplAI can specifically benefit your organization. Would you be available for a brief call next week to explore this further?

Thank you for considering this opportunity to enhance your compliance processes. I look forward to your response.

Best regards,

[Your Name]  
[Your Title]  
ComplAI  
[Your Phone Number]  
[Your Email Address]  
[Your LinkedIn Profile] (if applicable)  


----------------------------------------------------------------------------------------------------



Subject: Tired of Compliance Headaches? Let Us Handle the Heavy Lifting! 🏋️‍♂️

Hey [Recipient's Name],

Ever feel like navigating SOC2 compliance is like trying to read an IKEA manual? You know there’s a piece missing, and you’re pretty sure the instructions are in Swedish. 

Fear not! At ComplAI, we specialize in making compliance as easy as pie—without the weird crust thing going on. 🎉 Our AI-powered tool guides you through every step, ensuring you’re ready for audits without the stress of sleepless nights (or existential crises).

Imagine this: no more hunting for documents like they’re Pokémon, no more frantic meetings filled with “What do we do now?” Instead, you’ll have a sleek, streamlined process that practically runs itself. 

Are you ready to turn compliance chaos into calm? I’d love to set up a quick chat and show you how we can get you on the right track while keeping things entertaining. (Compliance can be fun—it's like a scavenger hunt, minus the muddy shoes!)

Let me know what time works for you, and we’ll make magic happen. 

Best,  
[Your Name]  
[Your Job Title]  
ComplAI  
[Your Contact Information]  

P.S. No IKEA assembly required! 😉


----------------------------------------------------------------------------------------------------



Subject: Streamline Your SOC 2 Compliance

Hi [Recipient's Name],

I hope this message finds you well. I wanted to introduce you to ComplAI, a SaaS solution designed to simplify SOC 2 compliance and audit preparation.

Our AI-powered platform automates documentation and tracks necessary controls, saving your team time and reducing compliance risks. Companies that use ComplAI have seen a significant decrease in audit preparation time and improved accuracy.

Would you be open to a brief call next week to discuss how we can support your compliance efforts?

Best,  
[Your Name]  
[Your Position]  
ComplAI  
[Your Phone Number]  
[Your LinkedIn Profile]  


----------------------------------------------------------------------------------------------------



In [13]:
# sales picker
decision = """
You pick the best cold sales email from the given options.
Imagine you are a customer and pick the one you are most likely to respond to.
Do not give an explanation; reply with the selected email only
"""

sales_picker = Agent(name='sales_picker', instructions=decision, model=MODEL_NAME)

In [14]:
message = "Write a cold sales email"

with trace("Sales selection workflow"):
    results = await asyncio.gather(
        Runner.run(sales_agent1, message),
        Runner.run(sales_agent2, message),
        Runner.run(sales_agent3, message)
    )

    outputs = [result.final_output for result in results]

    emails = "Cold sales emails:\n\n" + "\n\nEmail:\n\n".join(outputs)

    best = await Runner.run(sales_picker, emails)

In [15]:
display(Markdown(best.final_output))

Subject: Shield Your Business Like a Pro (and Maybe Save a Few Gray Hairs!)

Hey [Recipient's Name],

If you ever pondered what’s easier than teaching a cat to fetch, it’s definitely achieving SOC2 compliance! 😸

At ComplAI, we’ve brewed a SaaS tool that takes the ‘ugh’ out of audits and compliance. Our AI-powered platform helps you glide through the looping maze of SOC2 requirements without feeling like you’re on a roller coaster ride blindfolded. 🎢

Picture this: No more late nights diving into spreadsheets or endless emails chasing down your team for missing docs. Instead, you’ll have a compliance setup so smooth, you might just hear a *cha-ching* from the efficiency bank! 💰

Let’s chat for a few minutes. I promise to keep our conversation as refreshing as a lemonade on a hot day (with fewer calories, of course).

Looking forward to saving your sanity!

Cheers,  
[Your Name]  
[Your Job Title]  
ComplAI  
[Your Phone Number]  
[Your Email]  

P.S. Even if you don’t bite, I’ll still send over an amusing meme about compliance. It's all fun and games until someone forgets the audit! 😄

### Adding a tool to the mix

In [16]:
@function_tool 
def send_email_tool(subject:str, text_body: str, html_body: str) -> str:
    """
    Send out an email with the given subject and body to all sales prospects

    Args:
        subject: The subject of the email
        text_body: The body of the email as plain text
        html_body: The HTML body of the email
    """

    send_email(subject, text_body, html_body)
    return "Email Sent Successfully"

In [17]:
decision = """
You pick the best cold sales email from the given options.
Imagine you are a customer and pick the one you are most likely to respond to.
Then use your tool to send the email.
"""

required_tools = ModelSettings(tool_choice="required")

sales_sender = Agent(name='Sales Sender', instructions=decision, model=MODEL_NAME, tools=[send_email_tool], model_settings=required_tools)

In [18]:
message = "Write a cold sales email"

with trace("Sales selection workflow with sending"):
    results = await asyncio.gather(
        Runner.run(sales_agent1, message),
        Runner.run(sales_agent2, message),
        Runner.run(sales_agent3, message)
    )
    outputs = [result.final_output for result in results]

    emails = "Cold sales emails:\n\n" + "\n\nEmail:\n\n".join(outputs)

    response = await Runner.run(sales_sender, emails)
    print(f"Final response:\n{best.final_output}")

Final response:
Subject: Shield Your Business Like a Pro (and Maybe Save a Few Gray Hairs!)

Hey [Recipient's Name],

If you ever pondered what’s easier than teaching a cat to fetch, it’s definitely achieving SOC2 compliance! 😸

At ComplAI, we’ve brewed a SaaS tool that takes the ‘ugh’ out of audits and compliance. Our AI-powered platform helps you glide through the looping maze of SOC2 requirements without feeling like you’re on a roller coaster ride blindfolded. 🎢

Picture this: No more late nights diving into spreadsheets or endless emails chasing down your team for missing docs. Instead, you’ll have a compliance setup so smooth, you might just hear a *cha-ching* from the efficiency bank! 💰

Let’s chat for a few minutes. I promise to keep our conversation as refreshing as a lemonade on a hot day (with fewer calories, of course).

Looking forward to saving your sanity!

Cheers,  
[Your Name]  
[Your Job Title]  
ComplAI  
[Your Phone Number]  
[Your Email]  

P.S. Even if you don’t b

## Orchestrating by LLMs

#### A: via Tools

The simplest way to have 1 Agent choose to invoke another is by treating it as a tool call.  
The OpenAI Agents SDK gives a very simple way to do this.  

This works best when the flow is:  
Agent A -> Agent B -> Agent A  
And for the classic "Planning Agent" situation.

In [19]:
description = "Use this tool to write a sales email. In the input, just instruct it to write a sales email."

tool1 = sales_agent1.as_tool(tool_name='sales_email_writer_1', tool_description=description)
tool1

FunctionTool(name='sales_email_writer_1', description='Use this tool to write a sales email. In the input, just instruct it to write a sales email.', params_json_schema={'description': 'Default input schema for agent-as-tool calls.', 'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'AgentAsToolInput', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x000001D5601B4A40>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False, custom_data_extractor=None, allowed_callers=None, output_json_schema=None)

#### So now we can gather all the tools togther:

A tool for each of our 3 email-writing agents  
And a tool for our function to send emails

In [20]:
description = "Use this tool to write a sales email. In the input, just instruct it to write a sales email."

tool1 = sales_agent1.as_tool(tool_name='sales_email_writer_1', tool_description=description)
tool2 = sales_agent2.as_tool(tool_name='sales_email_writer_2', tool_description=description)
tool3 = sales_agent3.as_tool(tool_name='sales_email_writer_3', tool_description=description)

tools = [tool1, tool2, tool3, send_email_tool]
tools

[FunctionTool(name='sales_email_writer_1', description='Use this tool to write a sales email. In the input, just instruct it to write a sales email.', params_json_schema={'description': 'Default input schema for agent-as-tool calls.', 'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'AgentAsToolInput', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x000001D560ED3440>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False, custom_data_extractor=None, allowed_callers=None, output_json_schema=None),
 FunctionTool(name='sales_email_writer_2', description='Use this tool to write a sales email. In the input, just instruct it to write a sales email.', params_json_schema={'description': 'Default input

In [21]:
# Now it's time for our sales manager -- our planning agent
instructions = """
You are a Sales Manager at ComplAI.

Your job is to create and send ONE cold sales email.

Follow this exact workflow:

1. Call sales_email_writer_1 exactly once.
2. Call sales_email_writer_2 exactly once.
3. Call sales_email_writer_3 exactly once.
4. Compare the three generated emails.
5. Select the single best email.
6. Call send_email_tool exactly once with ONLY the selected email.
7. After sending the email, stop. Do not call any other tools.

Do not call any sales writer more than once.
Do not send more than one email.
"""

task = """
Generate three different sales email drafts using the three sales email
writer tools.

Then select the best draft and send ONLY that draft using send_email_tool.
"""

sales_manager = Agent(name="Sales Manager", instructions=instructions, tools=tools, model=MODEL_NAME)

In [22]:
with trace("Sales Manager"):
    result = await Runner.run(sales_manager, task)

## Structured Output

An LLM produces text in natural language. But we can have it instead produce a "Python Object".

1. We specify a Python Object
2. In the system prompt, the LLM is instructed to respond in JSON and follow a Schema which represents the Python Object.
3. The LLM outputs JSON, and the framework populates a Python Object based on it.

When we specify the Python object, we create a subclass of BaseModel, which is part of the Pydantic framework.  
Pydantic is a framework that easily allows defining a JSON schema and mapping between Python and json.

Notes:  
For more information research "constrained decoding".  
Not all providers support structured outputs.

In [24]:
from pydantic import BaseModel, Field

class EmailReview(BaseModel):
    is_professional: bool = Field(description="Whether the eamil is professional and appropriate")
    number_of_sentence: int = Field(description="The number of sentences in the body of the email, not including the greetings and signature")
    contains_placeholders: bool = Field(description="Whether the email contains placeholders for personalization")

In [25]:
EmailReview.model_json_schema()

{'properties': {'is_professional': {'description': 'Whether the eamil is professional and appropriate',
   'title': 'Is Professional',
   'type': 'boolean'},
  'number_of_sentence': {'description': 'The number of sentences in the body of the email, not including the greetings and signature',
   'title': 'Number Of Sentence',
   'type': 'integer'},
  'contains_placeholders': {'description': 'Whether the email contains placeholders for personalization',
   'title': 'Contains Placeholders',
   'type': 'boolean'}},
 'required': ['is_professional',
  'number_of_sentence',
  'contains_placeholders'],
 'title': 'EmailReview',
 'type': 'object'}

In [26]:
# Test
email = """
Hi [first_name],

I'm hitting you up to see if you'd like to buy our product. It's really great. You'll miss out if you don't buy it.

Laters.

sada
"""

In [27]:
checker = Agent(name='Checker', instructions='You review potential sales emails', model='gpt-4o-mini', output_type=EmailReview)
result = await Runner.run(checker, email)

review = result.final_output
review

EmailReview(is_professional=False, number_of_sentence=3, contains_placeholders=True)

## Guardrails
Guardrails are extremely important in AgenticAI. Put simply, they are controls that you code either in logic or with another LLM call, to prevent undesirable behavior.  

**Workflow boundaries**  
Guardrails are attached to agents and tools, but they do not all run at the same points in a workflow:
- **Input guardrails** run only for the first agent in the chain.
- **Output guardrails** run only for the agent that produces the final output.
- **Tool guardrails** run on every guarded function-tool invocation, including local MCP tools when their server configures guardrails, with input guardrails before execution and output guardrails after execution.



In [34]:
from agents import output_guardrail, GuardrailFunctionOutput

@output_guardrail
async def email_guardrail(ctx, agent, message):
    
    result = await Runner.run(checker, message, context=ctx.context)
    
    review = result.final_output
    is_problem = review.contains_placeholders or not review.is_professional

    return GuardrailFunctionOutput(output_info={"review": review}, tripwire_triggered=is_problem)

In [36]:
cowboy_instruction = instructions + "\nSpeak like a cowboy"

sales_agent_cowboy = Agent(name='Cowboy', instructions=cowboy_instruction, model='gpt-4o-mini', output_guardrails=[email_guardrail])

result = await Runner.run(sales_agent_cowboy, "Write a cold sales email")
print(result.final_output)

OutputGuardrailTripwireTriggered: Guardrail OutputGuardrail triggered tripwire

### On the other hand  

Guardrail without the built in functionality

In [38]:
simple_cowboy_instruction = intro + "\nSpeak like a cowboy"

simple_cowboy = Agent(name='Simple Cowboy', instructions=simple_cowboy_instruction, model='gpt-4o-mini')
result = await Runner.run(simple_cowboy, "Write a cold sales email")
email = result.final_output
display(Markdown(email))

Subject: Roundin’ Up SOC2 Compliance the Easy Way!

Howdy [Recipient's Name],

Hope this message finds ya well under the wide-open sky! I reckon you’re wranglin’ a few challenges with SOC2 compliance and audits. Well, saddle up, ‘cause I’ve got a trusty companion for ya—ComplAI!

Our SaaS tool is like a trusty steed, helpin’ you navigate the wild terrain of compliance with the power of AI. From prep to audit, we make sure you’re ridin’ smooth, so you can focus on ropin’ in them customers.

Let’s hitch up for a chat, and I’ll show ya how ComplAI can help your outfit ride the trail of compliance with ease.

Lookin’ forward to hearin’ from ya!

Best regards,  
[Your Name]  
[Your Title]  
ComplAI  
[Your Phone Number]  
[Your Email Address]  

In [39]:
result = await Runner.run(checker, email)
review = result.final_output
if not review.is_professional or review.contains_placeholders:
    print("The email is not professional or has placeholders and will not be sent")
else:
    print("Email looks good")

The email is not professional or has placeholders and will not be sent
